### Classification without RUS and SMOTE - (SVM, TSF, and GRU)

Norm       Train shape            Test shape              Class 0  Class 1  Imbalance ratio
──────────────────────────────────────────────────────────────────────────────────────────
per_file   (12455, 288, 10)       (3559, 288, 10)           12337      118           104.6x
per_row    (12455, 288, 10)       (3559, 288, 10)           12337      118           104.6x
no_norm    (12455, 288, 10)       (3559, 288, 10)           12337      118           104.6x

⚠️  Severe class imbalance detected (~105:1).
   Consider using class_weight='balanced' in SVM,
   and class_weight={0:1, 1:105} in GRU.


In [ ]:
# ══════════════════════════════════════════════════════════════
# BLOCK 2 — Reshape Helpers + SVM / TSF / GRU per Normalization
# Input shape: (samples, 288 timepoints, 10 channels)
# Primary metric: TSS (standard in space weather research)
# ══════════════════════════════════════════════════════════════
import numpy as np
import pandas as pd
import pickle
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, f1_score,
                             classification_report, confusion_matrix)
from sktime.classification.interval_based import TimeSeriesForestClassifier
from sktime.classification.compose import ColumnEnsembleClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout
from tensorflow.keras.utils import to_categorical
import warnings
warnings.filterwarnings("ignore")

# ── Class weight ratio (from Block 1: 12337 vs 118) ───────────
CLASS_WEIGHT_RATIO = 12337 / 118                        # ≈ 104.5
CLASS_WEIGHTS      = {0: 1.0, 1: CLASS_WEIGHT_RATIO}

# ── Reshape helpers ────────────────────────────────────────────
def to_svm(X):
    """(s, 288, 10) → (s, 2880) flattened"""
    return X.reshape(X.shape[0], -1)

def to_tsf(X):
    """(s, 288, 10) → (s, 10, 288)  [sktime: samples, channels, timepoints]"""
    return X.transpose(0, 2, 1)

def to_gru(X):
    """(s, 288, 10) → already correct for Keras GRU"""
    return X

# ── TSS metric ─────────────────────────────────────────────────
def tss_score(y_true, y_pred):
    """
    TSS = Recall(class 1) - False Alarm Rate
        = TP/(TP+FN) - FP/(FP+TN)
    Range: -1 to 1. Score of 0 = no skill. Standard in space weather.
    """
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    recall      = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    false_alarm = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    return recall - false_alarm

# ── Metric bundle helper ───────────────────────────────────────
def compute_metrics(y_true, y_pred):
    """Returns all reported metrics as a dict."""
    report = classification_report(y_true, y_pred,
                                   target_names=["Class 0", "Class 1"],
                                   output_dict=True)
    return {
        "accuracy":        round(accuracy_score(y_true, y_pred), 4),
        "macro_f1":        round(f1_score(y_true, y_pred, average="macro"), 4),
        "class1_f1":       round(report["Class 1"]["f1-score"],  4),
        "class1_recall":   round(report["Class 1"]["recall"],    4),
        "class1_precision":round(report["Class 1"]["precision"], 4),
        "tss":             round(tss_score(y_true, y_pred),      4),
        "cm":              confusion_matrix(y_true, y_pred),
        "report":          report,
    }

def print_metrics(metrics, clf_name):
    print(f"\n  ── [{clf_name}] ──────────────────────────────────────")
    print(f"  TSS:            {metrics['tss']:.4f}  ← primary metric")
    print(f"  Macro F1:       {metrics['macro_f1']:.4f}")
    print(f"  Class 1 F1:     {metrics['class1_f1']:.4f}")
    print(f"  Class 1 Recall: {metrics['class1_recall']:.4f}")
    print(f"  Class 1 Prec.:  {metrics['class1_precision']:.4f}")
    print(f"  Accuracy:       {metrics['accuracy']:.4f}  (misleading at 104:1)")
    tn, fp, fn, tp = metrics['cm'].ravel()
    print(f"  Confusion Matrix → TP:{tp}  FP:{fp}  TN:{tn}  FN:{fn}")

# ── Results store ──────────────────────────────────────────────
results = {}

# ══════════════════════════════════════════════════════════════
for norm_name, d in datasets.items():

    X_tr, y_tr = d["X_train"], d["y_train"]
    X_te, y_te = d["X_test"],  d["y_test"]

    n_samples, n_timepoints, n_channels = X_tr.shape
    n_classes = 2

    print(f"\n{'═'*60}")
    print(f"  NORMALIZATION: {norm_name.upper()}")
    print(f"{'═'*60}")

    norm_results = {}

    # ════════════════════════════════════════════════════════
    # [A]  SVM
    # ════════════════════════════════════════════════════════
    svm = SVC(kernel="rbf", C=1.0, class_weight="balanced", random_state=42)
    svm.fit(to_svm(X_tr), y_tr)
    svm_pred = svm.predict(to_svm(X_te))

    norm_results["SVM"] = compute_metrics(y_te, svm_pred)
    print_metrics(norm_results["SVM"], "SVM")

    # ════════════════════════════════════════════════════════
    # [B] TSF (Wrapped for Multivariate Data)
    # ════════════════════════════════════════════════════════
    # Create the base univariate classifier
    tsf_base = TimeSeriesForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    
    # Wrap it to handle all 10 channels (columns 0 through 9)
    estimators = [(f"tsf_channel_{i}", tsf_base, [i]) for i in range(10)]
    tsf = ColumnEnsembleClassifier(estimators=estimators)
    
    # Fit and Predict
    tsf.fit(to_tsf(X_tr), y_tr)
    tsf_pred = tsf.predict(to_tsf(X_te))

    norm_results["TSF"] = compute_metrics(y_te, tsf_pred)
    print_metrics(norm_results["TSF"], "TSF")

    # ════════════════════════════════════════════════════════
    # [C]  GRU
    # ════════════════════════════════════════════════════════
    y_tr_cat = to_categorical(y_tr, num_classes=n_classes)
    y_te_cat = to_categorical(y_te, num_classes=n_classes)

    tf.random.set_seed(42)
    gru_model = Sequential([
        GRU(64, input_shape=(n_timepoints, n_channels), return_sequences=True),
        Dropout(0.3),
        GRU(32, return_sequences=False),
        Dropout(0.3),
        Dense(32, activation="relu"),
        Dense(n_classes, activation="softmax")
    ], name=f"GRU_{norm_name}")

    gru_model.compile(optimizer="adam",
                      loss="categorical_crossentropy",
                      metrics=["accuracy"])

    gru_model.fit(
        to_gru(X_tr), y_tr_cat,
        epochs=30,
        batch_size=32,
        class_weight=CLASS_WEIGHTS,
        validation_split=0.1,
        verbose=0
    )

    gru_pred = np.argmax(gru_model.predict(to_gru(X_te), verbose=0), axis=1)
    norm_results["GRU"] = compute_metrics(y_te, gru_pred)
    print_metrics(norm_results["GRU"], "GRU")

    results[norm_name] = norm_results

# ══════════════════════════════════════════════════════════════
# BLOCK 3 — Summary Comparison Table
# ══════════════════════════════════════════════════════════════
rows = []
for norm_name, classifiers in results.items():
    for clf_name, metrics in classifiers.items():
        rows.append({
            "Normalization":   norm_name,
            "Classifier":      clf_name,
            "TSS":             metrics["tss"],           # ← ranked by this
            "Class 1 Recall":  metrics["class1_recall"],
            "Class 1 F1":      metrics["class1_f1"],
            "Macro F1":        metrics["macro_f1"],
            "Accuracy":        metrics["accuracy"],
        })

df_results = (pd.DataFrame(rows)
                .set_index(["Normalization", "Classifier"])
                .sort_values("TSS", ascending=False))

print("\n\n" + "═"*70)
print("  FINAL COMPARISON  (sorted by TSS — primary metric)")
print("═"*70)
print(df_results.to_string())

# ── Best normalization by average TSS across all 3 classifiers ─
best_norm_by_tss = (df_results.reset_index()
                              .groupby("Normalization")["TSS"]
                              .mean()
                              .idxmax())

best_single_row = df_results["TSS"].idxmax()
print(f"\n  ✓ Best single result:  {best_single_row[0]} / {best_single_row[1]}"
      f"  →  TSS = {df_results.loc[best_single_row, 'TSS']}")
print(f"  ✓ Best normalization overall (avg TSS): {best_norm_by_tss}")

# ══════════════════════════════════════════════════════════════
# NOTEBOOK 3 HANDOFF
# ══════════════════════════════════════════════════════════════
best_norm_dataset = datasets[best_norm_by_tss]

with open("./best_norm_for_nb3.pkl", "wb") as f:
    pickle.dump({
        "norm_name": best_norm_by_tss,
        "dataset":   best_norm_dataset,
        "results":   results,
        "df_summary": df_results,
    }, f)

print(f"\n  ✓ Saved to ./best_norm_for_nb3.pkl")
print(f"  ✓ Notebook 3 will use: '{best_norm_by_tss}' normalization")